# ARC-AGI-3 SG Baseline — P100 benchmark

Self-contained notebook for running `scripts/benchmark.py` against the 25 demo games on a Kaggle P100.

## Before you run

1. **Notebook settings (right pane)**
   - Accelerator: **GPU P100**
   - Internet: **ON** (Edit-mode interactive run is allowed to use ARC API)
2. **Secrets** (Add-ons → Secrets)
   - `ARC_API_KEY` — from <https://three.arcprize.org/user>
   - `GITHUB_TOKEN` — only if cloning from a private repo
3. **Edit `REPO_URL` cell below** to point at your fork

Total runtime at default budget (15 min × 5 games) ≈ **80 minutes**. To extend to all 25 games, raise `--budget-min` and `--games`.

## 1. Bootstrap: clone repo, install deps, override torch to CUDA

In [ ]:
REPO_URL = "https://github.com/<YOUR_USER>/arc-agi-3-pre.git"  # <-- EDIT ME
REPO_DIR = "/kaggle/working/arc-agi-3-pre"
PRIVATE_REPO = False  # set True if your repo is private and you set GITHUB_TOKEN

import os, subprocess, shlex, sys, shutil

# Idempotent: if cwd is inside REPO_DIR (from a prior run), escape first so
# rmtree / fetch don't pull the rug out from under git's cwd lookup.
os.chdir("/kaggle/working")

if PRIVATE_REPO:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth_url = REPO_URL.replace("https://", f"https://{token}@")
else:
    auth_url = REPO_URL

if os.path.exists(REPO_DIR):
    # Already cloned: refresh to latest main (preserves /kaggle/working/runs/ if any)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(shlex.split(f"git clone --depth 1 {auth_url} {REPO_DIR}"), check=True)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
subprocess.run(["git", "log", "--oneline", "-5"], check=True)

In [ ]:
# P100 has Pascal sm_60 — Kaggle's preinstalled torch 2.10+cu128 was built only
# for sm_70+ and will fail with "no kernel image is available for execution on
# the device" on the very first conv. Force-reinstall a torch wheel that still
# ships sm_60 kernels (cu121, torch <= 2.5.x).
#
# NOTE: each `!python -c "..."` MUST be one line — Jupyter treats `!cmd` as
# single-line magic and a multi-line string here triggers a SyntaxError.

!pip uninstall -q -y torch torchvision torchaudio 2>/dev/null
!pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q arc-agi==0.9.8 tensorboard

!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-'); print('arch_list:', torch.cuda.get_arch_list())"
!python -c "import torch, torch.nn as nn; x = torch.zeros(1, 16, 64, 64, device='cuda'); y = nn.Conv2d(16, 32, 3, padding=1).cuda()(x); print('conv2d on cuda OK, shape =', tuple(y.shape))"
!python -c "import arc_agi, arcengine; print('arc-agi:', getattr(arc_agi, '__version__', '?'), 'arcengine:', getattr(arcengine, '__version__', '?'))"

## 2. Wire ARC_API_KEY from Kaggle Secrets into the env

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["ARC_API_KEY"] = UserSecretsClient().get_secret("ARC_API_KEY")
print("ARC_API_KEY length:", len(os.environ["ARC_API_KEY"]), "(should be > 0)")

## 3. Quick GPU smoke (single game, 2000 steps)

Establishes that CUDA path works and gives an fps anchor before committing to the long run.

In [ ]:
!python scripts/train_sg.py --game cn04 --device cuda --steps 2000 --log-every 100 --seed 42

## 4. Multi-game benchmark

Default below: **5 games × 15 min** ≈ 80 min total. Edit budget/games to taste.
All output lands under `/kaggle/working/runs/<tag>/` and is downloadable from the Output panel.

If you have time budget (Kaggle gives 12h max per session), bump `--budget-min 60` and drop `--games` to play all 25.

In [ ]:
# Short first pass: 5 representative games × 15 min (~80 min total)
!python scripts/benchmark.py \
    --device cuda \
    --budget-min 15 \
    --games cn04,ft09,m0r0,lp85,r11l \
    --tag p100-short \
    --log-every 200

## 5. Show results inline + zip for download

In [ ]:
import json, pathlib
results_path = pathlib.Path("runs/p100-short/results.json")
if results_path.exists():
    print(json.dumps(json.loads(results_path.read_text()), indent=2))
scorecard_path = pathlib.Path("runs/p100-short/scorecard.json")
if scorecard_path.exists():
    sc = json.loads(scorecard_path.read_text())
    print("\n=== Scorecard total:", sc.get("score"))
    for env_entry in sc.get("environments", []):
        for run in env_entry.get("runs", []):
            print(f"  {env_entry['id']:30s}  state={run['state']:13s} actions={run['actions']:5d}  levels={run['levels_completed']}/{len(run['level_baseline_actions'])}  scores={[round(s,3) for s in run['level_scores']]}")

In [ ]:
# Zip everything for download (Output panel will show /kaggle/working/runs.zip)
!cd /kaggle/working && zip -qr runs.zip arc-agi-3-pre/runs && ls -lh runs.zip